# Day 5：拆头之后，怎样汇集内容并接回主干？

配套 [手算讲义](../docs/ATTENTION_HAND_CALC.md)，符号见 [统一规范](../docs/NOTATION.md)。
CPU FP64局部实验，不下载模型。直接给投影后的Q/K/V，不假装它们由下文残差示例X算出。
固定B=1、E=3、Hq=2、Hkv=1、D_k=D_v=2、T_q=1、T_kv=2。
暂不加入Norm、RoPE、dropout、FFN或真实Cache更新。形状类似最后一个位置查询两个可见位置。
本实验属于Day 5；当前学习位置见 [进度记录](../docs/LEARNING_PROGRESS.md)。本例验证计算，不验证模型能力或GPU速度。

In [ ]:
import math

import torch
from torch.nn import functional as F

dtype = torch.float64
a = math.sqrt(2) * math.log(3)
query = torch.tensor([[[[a, 0.]], [[0., a]]]], dtype=dtype)
compact_key = torch.tensor([[[[1., 0.], [0., 1.]]]], dtype=dtype)
compact_value = torch.tensor([[[[10., 0.], [0., 20.]]]], dtype=dtype)
batch, heads, query_length, key_dim = query.shape
kv_heads, kv_length, value_dim = compact_value.shape[1:]
assert heads % kv_heads == 0
assert compact_key.shape == (batch, kv_heads, kv_length, key_dim)
key = compact_key.repeat_interleave(heads // kv_heads, dim=1)
value = compact_value.repeat_interleave(heads // kv_heads, dim=1)
assert query.shape == (1, 2, 1, 2)
assert compact_key.shape == compact_value.shape == (1, 1, 2, 2)
print('Q:', tuple(query.shape))
print('compact K/V:', tuple(compact_key.shape), tuple(compact_value.shape))
print('教学复制后的K/V:', tuple(key.shape), tuple(value.shape))

## 1. Q/K决定读哪里

先预测：两头查询同一组K，概率会完全一样吗？
a=sqrt(2)*ln(3)，缩放后分数分别是`[ln(3),0]`与`[0,ln(3)]`。
对Key轴softmax得到3:1或1:3的概率，而不是跨head轴归一化。

In [ ]:
score = (query @ key.transpose(-2, -1)) / math.sqrt(key_dim)
probability = score.softmax(dim=-1)
expected_probability = torch.tensor([[[[0.75, 0.25]], [[0.25, 0.75]]]], dtype=dtype)
torch.testing.assert_close(probability, expected_probability)
torch.testing.assert_close(probability.sum(-1), torch.ones_like(probability.sum(-1)))
assert score.shape == probability.shape == (1, 2, 1, 2)
print('S shape:', tuple(score.shape), '各头分数:', score[0, :, 0].tolist())
print('P shape:', tuple(probability.shape), '各头权重:', probability[0, :, 0].tolist())

## 2. P/V决定读出什么

V位置0=[10,0]，V位置1=[0,20]。先手算每个头的加权结果，再执行。
Key位置轴被加权求和，留下每个Query的D_v维内容；不是把头平均。

In [ ]:
head_output = probability @ value
expected_heads = torch.tensor([[[[7.5, 5.]], [[2.5, 15.]]]], dtype=dtype)
torch.testing.assert_close(head_output, expected_heads)
assert head_output.shape == (batch, heads, query_length, value_dim)
# Both keys are visible to this final-position query; do not use an upper-left causal mask.
allowed = torch.ones(query_length, kv_length, dtype=torch.bool)
reference = F.scaled_dot_product_attention(
    query, key, value, attn_mask=allowed, dropout_p=0.0,
)
torch.testing.assert_close(head_output, reference)
merged = head_output.transpose(1, 2).reshape(batch, query_length, heads * value_dim)
torch.testing.assert_close(merged, torch.tensor([[[7.5, 5., 2.5, 15.]]], dtype=dtype))
print('每头输出:', head_output[0, :, 0].tolist(), tuple(head_output.shape))
print('合并输出:', merged.tolist(), tuple(merged.shape))
print('SDPA forward alignment: passed')

## 3. 输出投影：混合不同头，并接回E维主干

本例E=3，合并宽度Hq*D_v=4。人为选W_O的三行分别计算`c0+c2`、`c1+c3`、`c0-c2`。
先预测A，再与示例X=[1,2,3]逐元素相加。X仅为残差接口演示，不是上面Q/K/V的生成来源。
真实模型的W_O是可学习权重；即使输入输出同宽也可保留，作用不只是截短数组。

In [ ]:
output_weight = torch.tensor([
    [1., 0., 1., 0.],
    [0., 1., 0., 1.],
    [1., 0., -1., 0.],
], dtype=dtype)
update = merged @ output_weight.T
residual = torch.tensor([[[1., 2., 3.]]], dtype=dtype)
result = residual + update
assert output_weight.shape == (3, heads * value_dim)
assert residual.shape == update.shape == result.shape == (1, 1, 3)
torch.testing.assert_close(update, torch.tensor([[[10., 20., 5.]]], dtype=dtype))
torch.testing.assert_close(result, torch.tensor([[[11., 22., 8.]]], dtype=dtype))
print('W_O [out,in]:', tuple(output_weight.shape))
print('Attention更新A:', update.tolist())
print('X+A:', result.tolist())

## 4. 两个常见错误，主动构造负例

先预测：4维merged可否直接与3维X相加？
T_q=1可能掩盖合并前漏transpose的问题，所以另构造两个Query、两个head的标签张量。
数字只是布局标签，不是另一组Attention输出。

In [ ]:
try:
    residual + merged
except RuntimeError:
    print('预期负例：4维合并结果不能直接与3维主干相加。')
else:
    raise AssertionError('Residual feature widths must match.')
layout = torch.tensor([[[[1, 2], [3, 4]], [[5, 6], [7, 8]]]])
correct_merge = layout.transpose(1, 2).reshape(1, 2, 4)
incorrect_merge = layout.reshape(1, 2, 4)
assert correct_merge.tolist() == [[[1, 2, 5, 6], [3, 4, 7, 8]]]
assert incorrect_merge.tolist() == [[[1, 2, 3, 4], [5, 6, 7, 8]]]
assert not torch.equal(correct_merge, incorrect_merge)
print('正确先换轴:', correct_merge.tolist())
print('错误直接reshape:', incorrect_merge.tolist())

## 5. 一次只改一个量

实验A：让头1的Query与头0相同，K/V不变。先预测概率与A。
实验B：所有Value翻倍，Q/K不变。预测P、A、Y分别如何变。
注意Y=X+A中X没有翻倍；不能把更新量的线性变化直接当成残差结果整体翻倍。

In [ ]:
same_query = query[:, :1].repeat(1, heads, 1, 1)
same_probability = ((same_query @ key.transpose(-2, -1)) / math.sqrt(key_dim)).softmax(-1)
same_heads = same_probability @ value
same_merged = same_heads.transpose(1, 2).reshape(batch, query_length, -1)
same_update = same_merged @ output_weight.T
torch.testing.assert_close(same_heads[:, 0], same_heads[:, 1])
torch.testing.assert_close(same_update, torch.tensor([[[15., 10., 0.]]], dtype=dtype))
torch.testing.assert_close(residual + same_update, torch.tensor([[[16., 12., 3.]]], dtype=dtype))
doubled_heads = probability @ (2 * value)
doubled_merged = doubled_heads.transpose(1, 2).reshape(batch, query_length, -1)
doubled_update = doubled_merged @ output_weight.T
torch.testing.assert_close(doubled_update, 2 * update)
assert not torch.allclose(residual + doubled_update, 2 * result)
print('相同Q时A:', same_update.tolist(), 'Y:', (residual + same_update).tolist())
print('V翻倍时P不重算，A:', doubled_update.tolist(), 'Y:', (residual + doubled_update).tolist())

## 自测与参考答案

**1. 两个头共享K/V，输出一定一样吗？** 不一定，本例Q不同，概率不同，头输出分别为[7.5,5]、[2.5,15]。

**2. 分数表为什么没有D_k轴？** Q/K沿特征做点积，D_k轴求和；留下Query和Key位置轴。

**3. 输出为何有D_v轴？** 按Key位置汇集V，每个Value通道分别求和，得到每Query一个D_v维向量。

**4. 合并头和输出投影一样吗？** 不一样，前者无参数重排为Hq*D_v，后者用W_O学习跨头组合并输出E维。

**5. 输出投影只为降维吗？** 不是，即使输入输出同宽，也可学习不同头/特征的混合，不必等于单位映射。

**6. 为什么不能直接把本例merged加到X？** 最后一维分别为4和3；先投影回E=3才与主干匹配。

**7. 本例完成整个Block了吗？** 没有。Q/K/V直接给定，X只演示残差；没有Norm/RoPE/FFN/实际Cache/训练质量或性能验收。

**8. 下一步为什么学m/l/o？** 本例显式保存P再乘V；Online Softmax按Key块累积等价汇集结果，避免为最终输出保留整张P。

继续 [Day 6两块数字](day06_online_softmax.ipynb)，先理解数学，再区分融合GPU实现的性能收益。